# From Kronecker Delta to Dirac Delta — the grid, by hand

Companion notebook to the blog post [*From Kronecker Delta to Dirac Delta*](https://jen1995.github.io/posts/from-kronecker-to-dirac/) (Cohomology Zero, the quantum road).

Everything in the post is a finite computation on a grid followed by a limit. Here the finite computations are actually run, and the limits are watched numerically:

1. a state on a grid, in grid kets and in cell states;
2. the dead end — the inner product with the kernel $\delta_{ij}$ drains to zero — next to the fix, $\delta_{ij}/\Delta x$;
3. the requirement $\langle x_i|\psi_\Delta\rangle \to a(x)$ checked on finer and finer grids;
4. the kernel tested against a function: sifting as a limit of numbers;
5. pointwise versus weak convergence of the spikes;
6. one ket in three bases: a delta, a plane wave, a divergent column.

Only `numpy` and `matplotlib` are needed.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

INK, SEC, GREY = "#0b0b0b", "#52514e", "#8a8983"
BLUE, ORANGE, VIOLET, AQUA, GOLD = "#2a78d6", "#eb6834", "#4a3aa7", "#1baf7a", "#c98800"
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "font.size": 11})

# two continuous, square-integrable wavefunctions on the line (not normalized; that is irrelevant here)
def a(x):
    return np.exp(-x**2 / 1.3) * (1 + 0.3 * np.sin(2.1 * x))

def b(x):
    return np.exp(-(x - 0.4)**2 / 0.9) * np.cos(1.3 * x)

def grid(dx, L=6.0):
    """Grid points x_i a spacing dx apart on [-L, L]; cell i is [x_i - dx/2, x_i + dx/2]."""
    return np.arange(-L + dx / 2, L, dx)

x_fine = np.linspace(-6, 6, 4001)
exact_inner = np.trapezoid(np.conj(a(x_fine)) * b(x_fine), x_fine)
print(f"∫ a* b dx (fine quadrature) = {exact_inner:.6f}")

## 1. A state on a grid

The post writes the discretized state two ways: as the Riemann sum $\sum_i a(x_i)\,|x_i\rangle\,\Delta x$ over grid kets, and as $\sum_i c_i\,|\hat x_i\rangle$ over the orthonormal **cell states** — flat bumps of height $1/\sqrt{\Delta x}$ on disjoint cells, with $c_i = a(\xi_i)\sqrt{\Delta x}$. The two agree once $|x_i\rangle = |\hat x_i\rangle/\sqrt{\Delta x}$. Below, the cell-state expansion is drawn as the step function it is.

In [ ]:
def cell_state(xs, xi, dx):
    """Wavefunction of the cell state |x̂_i⟩: 1/sqrt(dx) on cell i, zero elsewhere."""
    return (np.abs(xs - xi) <= dx / 2) / np.sqrt(dx)

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4), sharey=True)
for ax, dx in zip(axes, [1.0, 0.5, 0.2]):
    xs = grid(dx)
    c = a(xs) * np.sqrt(dx)                         # c_i = a(x_i) sqrt(dx), sampling at the cell centre
    psi_step = sum(ci * cell_state(x_fine, xi, dx) for ci, xi in zip(c, xs))
    ax.plot(x_fine, a(x_fine), color=INK, lw=1.4, label="a(x)")
    ax.plot(x_fine, psi_step, color=BLUE, lw=1.4, label=r"$\sum_i c_i\,\hat x_i(x)$")
    ax.set_title(fr"$\Delta x = {dx}$", color=SEC); ax.set_xlim(-4, 4)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].legend(frameon=False)
plt.suptitle("the discretized state is a step function; it approaches a(x) as the grid is refined", color=SEC)
plt.tight_layout(); plt.show()

## 2. The dead end and the fix

With the guess $\langle x_i|y_j\rangle = \delta_{ij}$ the inner product of two discretized states is a Riemann sum times a leftover $\Delta x$, and it goes to zero. With the kernel the cell states force, $\delta_{ij}/\Delta x$, the leftover cancels and the sum converges to $\int a^* b\,dx$.

In [ ]:
dxs = np.array([2.0, 1.0, 0.5, 0.25, 0.125, 0.0625, 0.03125])
naive, fixed = [], []
for dx in dxs:
    xs = grid(dx)
    double_sum = np.sum(np.conj(a(xs)) * b(xs)) * dx * dx     # Σ_i Σ_j a* b δ_ij Δx Δy
    naive.append(double_sum)                                  # kernel δ_ij
    fixed.append(double_sum / dx)                             # kernel δ_ij / Δx

print(f"{'Δx':>8} {'kernel δ_ij':>14} {'kernel δ_ij/Δx':>16}")
for dx, n, f in zip(dxs, naive, fixed):
    print(f"{dx:8.5f} {n:14.6f} {f:16.6f}")
print(f"{'exact':>8} {'':>14} {exact_inner:16.6f}")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.semilogx(dxs, naive, "o-", color=ORANGE, label=r"kernel $\delta_{ij}$")
ax.semilogx(dxs, fixed, "o-", color=BLUE, label=r"kernel $\delta_{ij}/\Delta x$")
ax.axhline(exact_inner, color=GREY, ls="--", lw=1, label=r"$\int a^* b\,dx$")
ax.invert_xaxis(); ax.set_xlabel(r"$\Delta x$ (grid refined to the right)"); ax.set_ylabel(r"$\langle\psi_\Delta|\varphi_\Delta\rangle$")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 3. The requirement on finer grids

The one property asked of the position kets is $\langle x|\psi\rangle = a(x)$. On the grid, with the normalized kernel, $\langle x_i|\psi_\Delta\rangle = \sum_j a(x_j)\,\frac{\delta_{ij}}{\Delta x}\,\Delta x = a(x_i)$ exactly, and as the grid is refined the cell containing a fixed target $x$ shrinks onto it. With the naive kernel the same bracket returns $a(x_i)\,\Delta x \to 0$.

In [ ]:
x_target = 0.7
print(f"a(x) at the target x = {x_target}: {a(x_target):.6f}\n")
print(f"{'Δx':>8} {'x_i (cell centre)':>18} {'⟨x_i|ψ_Δ⟩, δ_ij/Δx':>20} {'⟨x_i|ψ_Δ⟩, δ_ij':>18}")
for dx in dxs:
    xs = grid(dx)
    i = np.argmin(np.abs(xs - x_target))              # the cell containing the target
    fixed_val = np.sum(a(xs) * (np.arange(len(xs)) == i) / dx * dx)   # Σ_j a(x_j) δ_ij/Δx · Δy
    naive_val = np.sum(a(xs) * (np.arange(len(xs)) == i) * dx)        # Σ_j a(x_j) δ_ij · Δy
    print(f"{dx:8.5f} {xs[i]:18.5f} {fixed_val:20.6f} {naive_val:18.6f}")

## 4. Sifting as a limit of numbers

Test the normalized kernel $K_\Delta(x_i, y_j) = \delta_{ij}/\Delta x$ against an arbitrary continuous $f$: the sum $\sum_j f(y_j)\,K_\Delta(x_i, y_j)\,\Delta y$ collapses to $f(x_i)$ and tends to $f(x)$. That is the sifting property, $\int f(y)\,\delta(x-y)\,dy = f(x)$, with the integral spelled out as the limit it stands for.

In [ ]:
def f(y):
    return np.sin(3 * y) * np.exp(-0.2 * y**2) + 0.5

for x_target in [-1.3, 0.0, 2.2]:
    print(f"target x = {x_target:5.2f}, f(x) = {f(x_target):.6f}")
    for dx in [1.0, 0.25, 0.0625, 0.015625]:
        ys = grid(dx)
        i = np.argmin(np.abs(ys - x_target))
        K = (np.arange(len(ys)) == i) / dx            # the spike: 1/Δx on cell i, 0 elsewhere
        tested = np.sum(f(ys) * K * dx)                # Σ_j f(y_j) K_Δ(x_i, y_j) Δy
        print(f"    Δx = {dx:9.6f}: Σ_j f(y_j) K_Δ Δy = {tested:.6f}   (|error| = {abs(tested - f(x_target)):.2e})")
    print()

## 5. Pointwise versus weak

Read the spike as a function of $y$: height $1/\Delta x$ on the cell around $x$, zero elsewhere. Pointwise, the values converge to no function — at $y = x$ they blow up, everywhere else they are eventually zero. Weakly, the integrals $\int f(y)\,K_\Delta(x, y)\,dy$ — averages of $f$ over the shrinking cell — converge to $f(x)$. The left panel plots the values, the right the integrals.

In [ ]:
x0 = 0.0
widths = [2.0, 1.0, 0.5, 0.25, 0.125, 0.0625]
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 3.8))
for dx, col in zip(widths, [BLUE, AQUA, VIOLET, ORANGE, GOLD, "#d04a8c"]):
    spike = (np.abs(x_fine - x0) <= dx / 2) / dx
    axL.plot(x_fine, spike, color=col, lw=1.4, label=fr"$\Delta x={dx}$")
axL.set_xlim(-1.5, 1.5); axL.set_title("the spikes: height 1/Δx on a shrinking cell — no pointwise limit", color=SEC, fontsize=10)
axL.legend(frameon=False, fontsize=8); axL.spines[["top", "right"]].set_visible(False)

tested = [np.trapezoid(f(x_fine) * ((np.abs(x_fine - x0) <= dx / 2) / dx), x_fine) for dx in widths]
axR.semilogx(widths, tested, "o-", color=BLUE)
axR.axhline(f(x0), color=GREY, ls="--", lw=1)
axR.text(widths[0], f(x0), "  f(x)", color=SEC, va="bottom")
axR.invert_xaxis(); axR.set_xlabel(r"$\Delta x$"); axR.set_title(r"the integrals $\int f\,K_\Delta\,dy$ converge to f(x)", color=SEC, fontsize=10)
axR.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 6. One ket, three bases

The same generalized vector $|x_0\rangle$ has different coordinates in different bases, each obtained as an inner product with the basis vectors that the delta reduces to a value at $x_0$:

- position basis: $\delta(y - x_0)$ — approximated here by the spike;
- momentum basis: $\langle k|x_0\rangle = e^{-ikx_0}/\sqrt{2\pi}$, constant modulus;
- energy basis of a box of length $L$: $\langle E_n|x_0\rangle = \varphi_n(x_0) = \sqrt{2/L}\,\sin(n\pi x_0/L)$, a discrete list whose squares do not decay — the infinite norm, seen as a divergent series.

Below, the box coordinates are computed two ways: directly as $\varphi_n(x_0)$, and as the inner product of $\varphi_n$ with a narrow spike at $x_0$. They agree as the spike narrows, and the partial sums $\sum_{n\le N}|\varphi_n(x_0)|^2$ grow without bound.

In [ ]:
L, x0 = 3.0, 1.1
ys = np.linspace(0, L, 6001)
def phi(n, y):
    return np.sqrt(2 / L) * np.sin(n * np.pi * y / L)

print(f"{'n':>3} {'φ_n(x0)':>10} {'⟨φ_n|spike⟩, Δx=0.1':>22} {'Δx=0.01':>10}")
for n in range(1, 9):
    direct = phi(n, x0)
    via = [np.trapezoid(phi(n, ys) * ((np.abs(ys - x0) <= dx / 2) / dx), ys) for dx in (0.1, 0.01)]
    print(f"{n:3d} {direct:10.5f} {via[0]:22.5f} {via[1]:10.5f}")

N = np.arange(1, 401)
partial = np.cumsum([phi(n, x0)**2 for n in N])
fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(N, partial, color=ORANGE, lw=1.5)
ax.set_xlabel("N"); ax.set_ylabel(r"$\sum_{n\leq N} |\varphi_n(x_0)|^2$")
ax.set_title("the squared coordinates of |x₀⟩ in the box basis: the partial sums diverge", color=SEC, fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

k = np.linspace(-10, 10, 400)
print(f"\n|⟨k|x0⟩| = 1/sqrt(2π) = {1/np.sqrt(2*np.pi):.5f} for every k — a plane wave of constant modulus:",
      np.allclose(np.abs(np.exp(-1j * k * x0) / np.sqrt(2 * np.pi)), 1 / np.sqrt(2 * np.pi)))

## What to try

- Replace `a` and `b` by other continuous, square-integrable functions; the dead end and the fix do not care which.
- Make `f` in section 4 discontinuous at the target point and watch the sifting sums stop converging: continuity was used.
- In section 5, sample the test function on a grid *coarser* than the spike (`Δy > Δx`) and see the sum miss the spike entirely — the direction of the coupling discussed in Appendix B of the post.